# Train T1 + domain classifier on Colab GPU

This offloads the two CPU-heavy parts of the fake-news-orchestration pipeline (T1 fine-tuning, the 3-encoder x 12-config domain-classifier grid search for Table 5) onto Colab's GPU. Everything LLM-dependent (T2, T3, the router) stays on your laptop since it talks to your local LM Studio server, which Colab can't reach.

**Before running:** in Colab's menu, go to `Runtime > Change runtime type` and select **T4 GPU**.

**Two ways to run this, pick one:**
- **Full run** (`main()`, cell below "Train"): fresh session, nothing trained yet. Trains T1 for all 3 seeds + the full grid search for all 3 domain-classifier encoders.
- **Resume just one encoder** (`train_domain_classifier_only()`, the cell after): you already have T1 and some encoders' checkpoints locally (e.g. from an earlier Colab run) and only need to fill in one more (e.g. `xlm-roberta-base`). This needs only the code + AMTCele data -- **no checkpoint upload required** -- and only downloads that one encoder's result instead of a multi-GB archive of everything.

**Steps:**
1. Run the cells in order up through "Install dependencies".
2. When prompted, upload `fake-news-orchestration.zip` (the project code + AMTCele data, no `.venv`).
3. Run **either** the "Train (full run)" cell **or** the "Train (resume one encoder)" cell, not both.
4. Download the resulting zip and unzip it into your local project so it lands as `fake-news-orchestration/checkpoints/...`, matching `configs/default.yaml`'s `paths.t1_checkpoint_dir` / `paths.domain_classifier_checkpoint_dir`. Any local script that supports it (`run_catalogue`, `run_main_veracity`, `run_ablations`, `run_routing_analysis`, `run_efficiency`, `run_domain_classification`) will auto-detect and load the checkpoint instead of retraining.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -- go set Runtime > Change runtime type > T4 GPU')

## Upload the project
Upload `fake-news-orchestration.zip` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/project')
%cd /content/project

## Install dependencies
Reuses the project's own `requirements.txt` so versions match your local venv. `openai` (API-only extra) and `bitsandbytes` aren't needed here -- nothing in this notebook calls an LLM or does local 4-bit inference.

In [ ]:
!pip install -q sentence-transformers rank-bm25 pyyaml faiss-cpu dateparser

## Train (full run)
Trains T1 once per seed in `configs/default.yaml`'s `seeds` list (reused by Table 2/6/8/10/11's scripts), then runs the full domain-classifier grid search for Table 5 across all 3 candidate encoders. Prints progress per epoch / per grid config, same as the local logging. **Only run this on a fresh session with nothing trained yet -- skip to the next cell instead if you're resuming.**

In [ ]:
from colab.train_on_colab import main
main()

## Train (resume one encoder) -- alternative to the cell above
Use this instead of `main()` when T1 and some domain-classifier encoders are already done locally and you only need to fill in the rest (default: `xlm-roberta-base`). Needs only the code + AMTCele data uploaded above -- does not require uploading any existing checkpoints, and does not touch T1 at all. Pass a list to train different/multiple encoders, e.g. `train_domain_classifier_only(["microsoft/deberta-v3-base", "xlm-roberta-base"])`.

In [ ]:
from colab.train_on_colab import train_domain_classifier_only
train_domain_classifier_only()  # defaults to ["xlm-roberta-base"]

## Package checkpoints for download
If you ran the **full run**, this zips everything (a few GB: 3 T1 seeds + 3 domain-classifier encoders) and saves to Google Drive first (more reliable than a direct browser download for large files), plus offers a direct download.

If you ran **resume one encoder**, edit the path below to `checkpoints/domain_classifier/<encoder-name>` so you only zip/download the one new encoder, not a redundant copy of everything.

In [ ]:
CHECKPOINT_PATH_TO_ZIP = 'checkpoints/'  # full run: 'checkpoints/'. Resume-one-encoder: e.g. 'checkpoints/domain_classifier/xlm-roberta-base'
!zip -qr checkpoints_out.zip {CHECKPOINT_PATH_TO_ZIP}
import os
print('checkpoints_out.zip size (MB):', round(os.path.getsize('checkpoints_out.zip') / 1e6))

In [ ]:
# Option A: save to Google Drive (recommended for multi-GB files)
from google.colab import drive
drive.mount('/content/drive')
!cp checkpoints_out.zip /content/drive/MyDrive/checkpoints_out.zip
print('Saved to Google Drive: MyDrive/checkpoints_out.zip')

In [ ]:
# Option B: direct browser download (can be flaky for very large files)
from google.colab import files
files.download('checkpoints_out.zip')